## Analysis of annotations 

This notebook does a first analysis of the annotation data, in order to create the first ML model predicting 'points d'arrêt'


In [1]:
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

/home/cnouri/miniconda3/envs/fb-pa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load your annotated dataframe
df_anno = pd.read_csv('../data/clean_annotated_comments.csv', low_memory=False)

# Inspect
print(df_anno.head())

            url_id                                          clean_url  \
0  4a5jagld15qh9m4  https://www.lenouveaudetective.com/dompierre-s...   
1  4a5jagld15qh9m4  https://www.lenouveaudetective.com/dompierre-s...   
2  4a5jagld15qh9m4  https://www.lenouveaudetective.com/dompierre-s...   
3  9kikqscza41dy1l  http://secretnews.fr/2018/01/25/ferrero-rappel...   
4  9kikqscza41dy1l  http://secretnews.fr/2018/01/25/ferrero-rappel...   

            parent_domain       source_type        theme  \
0  lenouveaudetective.com  sensationnaliste  fait divers   
1  lenouveaudetective.com  sensationnaliste  fait divers   
2  lenouveaudetective.com  sensationnaliste  fait divers   
3           secretnews.fr         parodique        santé   
4           secretnews.fr         parodique        santé   

   false_news_usr_feedback  hate_speech_usr_feedback  \
0                     79.0                      24.0   
1                     79.0                      24.0   
2                     79.0      

In [3]:
# Print the number of rows and columns
print(f"Number of rows: {df_anno.shape[0]}")
print(f"Number of columns: {df_anno.shape[1]}")

Number of rows: 43305
Number of columns: 22


In [4]:
# Iterate over each column and print its name and an example value
for col in df_anno.columns:
    example_value = df_anno[col].dropna().iloc[0] if not df_anno[col].dropna().empty else 'No data'
    print(f"Column: {col}")
    print(f"Example: {example_value}")
    print("-" * 40)

Column: url_id
Example: 4a5jagld15qh9m4
----------------------------------------
Column: clean_url
Example: https://www.lenouveaudetective.com/dompierre-sur-besbredisparition-alexis-voyer-facebook/
----------------------------------------
Column: parent_domain
Example: lenouveaudetective.com
----------------------------------------
Column: source_type
Example: sensationnaliste
----------------------------------------
Column: theme
Example: fait divers
----------------------------------------
Column: false_news_usr_feedback
Example: 79.0
----------------------------------------
Column: hate_speech_usr_feedback
Example: 24.0
----------------------------------------
Column: account_name
Example: AUX Portes DU Pouvoir
----------------------------------------
Column: moderation_charter
Example: no_moderation
----------------------------------------
Column: account_subscriber_count
Example: 7962.0
----------------------------------------
Column: page_group_type
Example: extrême droite
------

## Try ML model 

In [5]:
# Encode labels (if they are strings)
label_encoder = LabelEncoder()
df_anno['label_encoded'] = label_encoder.fit_transform(df_anno['stop'])

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df_anno[['text', 'label_encoded']])
dataset = dataset.train_test_split(test_size=0.2)

dataset = dataset.rename_column('label_encoded', 'labels')

In [6]:

model_name = "almanach/camembertv2-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
num_labels = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/cnouri/miniconda3/envs/fb-pa/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at almanach/camembertv2-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 8661/8661 [00:02<00:00, 2924.23 examples/s]


In [7]:
training_args = TrainingArguments(
    output_dir="./camembertv2_results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

In [8]:
trainer.train()

/home/cnouri/miniconda3/envs/fb-pa/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
model.save_pretrained('./camembert_text_classifier')
tokenizer.save_pretrained('./camembert_text_classifier')

In [ ]:
Column: text
Example: Pourquoi toutes ces disparitions de jeunes, c'est vraiment alarmant Si on ne les retrouvent pas, ou sont-ils Mon Dieu, j'ai un mauvais pré sentiment. Ca me rappelle les enlèvements de beaucoup de jeunes gens en Algérie pendant la guerre. C'est triste.🙏 😭 🤒
----------------------------------------
Column: stop
Example: no_stop